# 01 — Perfil Demográfico e Geográfico

**Objetivo**: analisar distribuição das variáveis demográficas e geográficas dos municípios.

**Inputs**: `data/processed/trusted_municipios_eda.parquet`.

**Outputs**: figuras demográficas e tabela resumo por região.

In [ ]:
# Imports
import logging

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import PROCESSED_DATA_DIR
from src.utils.eda import (
    compute_descriptive_stats,
    plot_boxplot_by_group,
    plot_distribution,
    plot_correlation_heatmap,
    save_json,
)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Leitura
df = pd.read_parquet(PROCESSED_DATA_DIR / "trusted_municipios_eda.parquet")
logger.info("Linhas: %d, Colunas: %d", df.shape[0], df.shape[1])

## 1. Estatísticas descritivas das variáveis demográficas

In [ ]:
demo_cols = [
    "populacao_total",
    "populacao_18_35_pct",
    "populacao_urbana_pct",
    "escolaridade_ensino_medio_pct",
    "idhm",
]
desc = compute_descriptive_stats(df[demo_cols])
display(desc)

## 2. Distribuições

In [ ]:
plot_distribution(
    df, "populacao_total", log_scale=True, filename="01_dist_populacao_total.png"
)
plt.show()

for col in [
    "populacao_18_35_pct",
    "populacao_urbana_pct",
    "escolaridade_ensino_medio_pct",
]:
    plot_distribution(df, col, filename=f"01_dist_{col}.png")
    plt.show()

## 3. Boxplots por região

In [ ]:
for col in ["escolaridade_ensino_medio_pct", "populacao_18_35_pct", "populacao_urbana_pct"]:
    plot_boxplot_by_group(df, col, "nome_regiao", filename=f"01_boxplot_{col}_regiao.png")
    plt.show()

## 4. Correlação entre variáveis demográficas

In [ ]:
plot_correlation_heatmap(
    df,
    ["populacao_18_35_pct", "populacao_urbana_pct", "escolaridade_ensino_medio_pct", "idhm"],
    method="spearman",
    title="Correlação entre variáveis demográficas",
    filename="01_correlacao_demografica.png",
)
plt.show()

## 5. Tabela resumo por região

In [ ]:
resumo_regiao = (
    df.groupby("nome_regiao")[demo_cols]
    .mean()
    .round(2)
    .sort_values("escolaridade_ensino_medio_pct", ascending=False)
)
display(resumo_regiao)

## 6. Municípios com perfil jovem + urbano + escolarizado

In [ ]:
mediana_jovens = df["populacao_18_35_pct"].median()
mediana_urbana = df["populacao_urbana_pct"].median()
mediana_escolar = df["escolaridade_ensino_medio_pct"].median()

df_perfil_alto = df[
    (df["populacao_18_35_pct"] > mediana_jovens)
    & (df["populacao_urbana_pct"] > mediana_urbana)
    & (df["escolaridade_ensino_medio_pct"] > mediana_escolar)
]

logger.info("Municípios com perfil alto: %d", len(df_perfil_alto))
display(df_perfil_alto[["nome_municipio", "sigla_uf", "populacao_total"]].head(10))

In [ ]:
report = {
    "resumo_por_regiao": resumo_regiao.to_dict(),
    "municipios_perfil_alto": int(len(df_perfil_alto)),
}
save_json(report, "01_perfil_demografico.json")